# Real Time Factor (RTF) Measurement

This notebook measures the Real Time Factor (RTF) of various RTF estimation methods defined in the project configuration. It iterates through the methods, runs predictions with a batch size of 1, and reports statistics (Min, Max, Mean, Median, Std) for each.

In [1]:
import os
import sys
import time
import yaml
import torch
import numpy as np
import lightning as pl
import gc
from lightning.pytorch.callbacks import Callback
from pathlib import Path

# Add src to python path to import project modules
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
src_path = os.path.join(project_root, "src")
if src_path not in sys.path:
    sys.path.append(src_path)

# Import project modules
from models import RTFmodule, OnlineCGMM_MVDR, BlockOnlineGSS
from datasets_local import (
    BrudexDataModule,
    ScenarioGenerationConfig,
    BrudexConfig,
)
from utilities import STFTtransform
import building_blocks.feature_extractors as bb_fe
import building_blocks.rtf_estimators as bb_rtf_e
import building_blocks.source_count_estimators as bb_sce

# Yaml constructors
def tuple_constructor(loader: yaml.SafeLoader, node: yaml.nodes.Node) -> tuple:
    if not isinstance(node, yaml.nodes.SequenceNode):
        raise yaml.constructor.ConstructorError(
            "expected a sequence node, but found %s" % type(node)
        )
    return tuple(loader.construct_sequence(node))

yaml.add_constructor("!tuple", tuple_constructor, Loader=yaml.SafeLoader)
yaml.add_constructor("!range", tuple_constructor, Loader=yaml.SafeLoader)

In [2]:
class RealTimeFactorCallback(Callback):
    def __init__(self):
        super().__init__()
        self.rtfs = []
        self.start_time = 0.0

    def on_predict_batch_start(self, trainer, pl_module, batch, batch_idx, dataloader_idx=0):
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        self.start_time = time.perf_counter()

    def on_predict_batch_end(self, trainer, pl_module, outputs, batch, batch_idx, dataloader_idx=0):
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        end_time = time.perf_counter()
        inference_time = end_time - self.start_time

        # Determine duration of the signal
        duration = 0.0
        samples = list(batch.meta["sad_samples"][0].values())[0].shape[0]
        fs = pl_module.transform.sampling_frequency if hasattr(pl_module, 'transform') else 16000
        duration = samples / fs
            
        if duration > 0:
            rtf = inference_time / duration
            self.rtfs.append(rtf)

    def print_stats(self, method_name):
        if len(self.rtfs) > 0:
            rtfs = np.array(self.rtfs)
            print(f"\n--- RTF Statistics for {method_name} ---")
            print(f"Min:    {np.min(rtfs):.6f}")
            print(f"Max:    {np.max(rtfs):.6f}")
            print(f"Mean:   {np.mean(rtfs):.6f}")
            print(f"Median: {np.median(rtfs):.6f}")
            print(f"Std:    {np.std(rtfs):.6f}")
            print(f"Count:  {len(rtfs)}")
            print("---------------------------------------\n")
        else:
            print(f"No RTF data collected for {method_name}.")

    def reset(self):
        self.rtfs = []

In [3]:
# --- Configuration ---
EXP_TYPE = "main_exp"  # "main_exp", "seglen", "deactivation"
EXP_COLLECTION = "J1_BXLS_" + EXP_TYPE
USE_DEBUG_DATASET = False # If True, uses a smaller dataset

exp_collection_paths = {
    "J1_BXLS_main_exp": {
        "dataset_config": "configs/datasets/J1_dataset_main_exp.yaml",
        "ebop_algorithm_config": "configs/algorithms/ebop.yaml",
        "framework_config": "configs/J1_framework_main_exp.yaml",
    },
    "J1_BXLS_seglen": {
        "dataset_config": "configs/datasets/J1_dataset_seglen.yaml",
        "ebop_algorithm_config": "configs/algorithms/J1_ebop_seglen.yaml",
        "framework_config": "configs/J1_framework_seglen.yaml",
    },
    "J1_BXLS_deactivation": {
        "dataset_config": "configs/datasets/J1_dataset_deactivation.yaml",
        "ebop_algorithm_config": "configs/algorithms/ebop.yaml",
        "framework_config": "configs/J1_framework_deactivation.yaml",
    },
    "J1_BXLS_deactivation_50ms": {
        "dataset_config": "configs/datasets/J1_dataset_deactivation_50ms.yaml",
        "ebop_algorithm_config": "configs/algorithms/J1_ebop_deactivation_50ms.yaml",
        "framework_config": "configs/J1_framework_deactivation_50ms.yaml",
    },
}

torch.set_float32_matmul_precision("highest")

# Load configurations
dataset_config_path = os.path.join(src_path, exp_collection_paths[EXP_COLLECTION]["dataset_config"])
ebop_algorithm_config_path = os.path.join(src_path, exp_collection_paths[EXP_COLLECTION]["ebop_algorithm_config"])
framework_config_path = os.path.join(src_path, exp_collection_paths[EXP_COLLECTION]["framework_config"])

config = {}
with open(dataset_config_path, "r") as f:
    config["dataset"] = yaml.safe_load(f)
with open(ebop_algorithm_config_path, "r") as f:
    config["ebop_algorithm"] = yaml.safe_load(f)
with open(framework_config_path, "r") as f:
    config["framework"] = yaml.safe_load(f)

# Override for RTF Measurement
config["framework"]["rtf_batch_size"] = 1 # Force batch size 1
config["framework"]["rtf_devices"] = [2] # Use specific GPU or cpu
config["framework"]["rtf_accelerator"] = "gpu" if torch.cuda.is_available() else "cpu"

if USE_DEBUG_DATASET:
    print("Using DEBUG mode for dataset.")
    config["dataset"]["id"] = config["dataset"]["id"] + "_debug"
    config["dataset"]["num_scenarios"] = [4, 4, 100] # Small number for testing

In [6]:
# --- Setup Data Module ---
stft_transform = STFTtransform(
    frame_length=config["ebop_algorithm"]["frame_length"],
    frame_shift=config["ebop_algorithm"]["frame_shift"],
    sampling_frequency=config["ebop_algorithm"]["sampling_frequency"],
    window_type=config["ebop_algorithm"]["window_type"],
)

scenario_config = ScenarioGenerationConfig(
    max_sources=config["dataset"]["max_sources"],
    signal_lengths=config["dataset"]["signal_lengths"],
    initial_noise_only_duration=config["dataset"]["initial_noise_only_duration"],
    snrs=config["dataset"]["snrs"],
    source_power_range=config["dataset"]["source_power_range"],
    time_between_events=config["dataset"]["time_between_events"],
    fix_seglen=config["dataset"].get("fix_seglen", False),
    activations_only=config["dataset"]["activations_only"],
    remove_silence=config["dataset"]["remove_silence"],
    neglect_silence4oracle_sa=config["dataset"]["neglect_silence4oracle_sa"],
    bridge_clean_speech_gaps=config["dataset"]["bridge_clean_speech_gaps"],
    vad_threshold2define_oracle=config["dataset"]["vad_threshold2define_oracle"],
    vad_threshold2select_clean_speech=config["dataset"]["vad_threshold2select_clean_speech"],
)

brudex_config = BrudexConfig(
    reverb_conditions=config["dataset"]["reverb_conditions"],
    microphone_arrays=config["dataset"]["microphone_arrays"],
    noise_types=config["dataset"]["noise_types"],
    doas=config["dataset"]["doas"],
)

# For RTF measurement, we primarily need the STFT features, so we can init with STFT extractor
stft_feat_extractor = bb_fe.STFT_Conv_Feature_Encoder(
    transform=stft_transform
)

dm = BrudexDataModule(
    id=config["dataset"]["id"],
    batch_size=1, # FORCE 1
    num_workers=config["framework"]["sad_num_workers"],
    num_scenarios=config["dataset"]["num_scenarios"],
    transform=stft_transform,
    sampling_frequency=config["ebop_algorithm"]["sampling_frequency"],
    generation_config=scenario_config,
    brudex_config=brudex_config,
    clean_speech_databases=config["dataset"]["clean_speech_databases"],
    feature_extractor=stft_feat_extractor,
    seed=config["dataset"]["seed"],
    acc_device=config["framework"].get("datagen_acc_device", "cpu"),
    reset=False, # Don't reset dataset generation for RTF measurement
)

dm.force_load_stft = True # Ensure we load STFT directly if implemented in DM
dm.setup("test")
# Use only the test set for benchmarking
test_loader = dm.test_dataloader()

Found implementation for clean speech database: Librispeech
LibrispeechDatabase initialized to use splits: ['train-clean-360', 'dev-clean', 'test-clean']
Found 12 RIR .mat files to consider.
Found 2 noise .mat files to consider.
Brudex Data Module Parameters:
  Batch Size: 1
  Num Workers: 0
  Num Scenarios: [5280, 528, 2640]
  STFTtransform Parameters:
    Frame Length: 0.064 s
    Frame Shift: 0.016 s
    Sampling Frequency: 16000 Hz
    Window Type: sqrt-hann
    NFFT: 1024 samples
    Hop Length: 256 samples
  Sampling Frequency: 16000 [Hz]
  Max Sources: 3
  Signal Lengths: 60.0 [s]
  Initial Noise Only Duration: (0.5, 5.0) [s]
  Reverb Times: ['low'] [ms]
  Microphone Arrays: ['BTE_IE']
  Train Noise Types: ['babble', 'cafeteria']
  Val Noise Types: ['babble', 'cafeteria']
  Test Noise Types: ['babble', 'cafeteria']
  SNRs: [0, 5, 10, 15] [dB]
  Clean Speech Databases: ['Librispeech']
  Train DOAs: [-150, -120, -90, -60, -30, 0, 30, 60, 90, 120, 150, 180] [°]
  Val DOAs: [-150, -

In [7]:
# --- Run Benchmark Loop ---
rtf_methods_to_test = config["ebop_algorithm"]["rtf_methods"]
# rtf_methods_to_test = ["BOP", "BOPO-W"] # Uncomment to test subset

print(f"Benchmarking methods: {rtf_methods_to_test}")

rtf_callback = RealTimeFactorCallback()

results = {}

for rtf_method_name in rtf_methods_to_test:
    print(f"\nPreparing {rtf_method_name}...")
    rtf_callback.reset()

    # --- Instantiate Model ---
    if rtf_method_name == "online-cgmm-mvdr":
        rtf_model = OnlineCGMM_MVDR(
            transform=stft_transform,
            max_sources=config["dataset"]["max_sources"],
            chunk_size=config["ebop_algorithm"].get("chunk_size", 0.064),
            batch_size=1,
            loss_config=config["framework"]["rtf_loss"],
            optimizer_config=config["framework"]["rtf_optimizer"],
            lr_scheduler_config=config["framework"]["rtf_lr_scheduler"],
            compute_complexity_metrics=False,
            check_causality=False,
        )
    elif rtf_method_name == "block-online-GSS":
        rtf_model = BlockOnlineGSS(
            transform=stft_transform,
            max_sources=config["dataset"]["max_sources"],
            block_size=config["ebop_algorithm"]["gss_block_length"],
            pre_context=config["ebop_algorithm"]["gss_pre_context"],
            latency_constraint=config["ebop_algorithm"].get("gss_latency_constraint", True),
            batch_size=1,
            loss_config=config["framework"]["rtf_loss"],
            optimizer_config=config["framework"]["rtf_optimizer"],
            lr_scheduler_config=config["framework"]["rtf_lr_scheduler"],
            compute_complexity_metrics=False,
            check_causality=False,
        )
    elif rtf_method_name == "oracle":
        # Oracle usually doesn't involve computation, but we can measure overhead
        rtf_estimator = bb_rtf_e.Oracle()
        rtf_model = RTFmodule(
                transform=stft_transform,
                smoothing_time_constant=config["ebop_algorithm"]["smoothing_time_constant"],
                segment_forgetting_factor=config["ebop_algorithm"]["segment_forgetting_factor"],
                fix_prev_rel_time=config["ebop_algorithm"]["fix_prev_rel_time"],
                noisy_cov_init_time=config["ebop_algorithm"]["noisy_cov_init_time"],
                registry_HA_threshold=config["ebop_algorithm"]["registry_HA_threshold"],
                rtf_estimator=rtf_estimator,
                max_sources=config["dataset"]["max_sources"],
                interferer_gain=config["ebop_algorithm"]["interferer_gain"],
                source_activity_method=None, # Use Oracle from batch
                batch_size=1,
                loss_config=config["framework"]["rtf_loss"],
                optimizer_config=config["framework"]["rtf_optimizer"],
                lr_scheduler_config=config["framework"]["rtf_lr_scheduler"],
                compute_complexity_metrics=False,
                check_causality=False,
            )
    else:
         # Standard RTF Estimators
        method_class_name = rtf_method_name.replace("-", "_")
        if hasattr(bb_rtf_e, method_class_name):
            rtf_estimator = getattr(bb_rtf_e, method_class_name)()
            
            rtf_model = RTFmodule(
                transform=stft_transform,
                smoothing_time_constant=config["ebop_algorithm"]["smoothing_time_constant"],
                segment_forgetting_factor=config["ebop_algorithm"]["segment_forgetting_factor"],
                fix_prev_rel_time=config["ebop_algorithm"]["fix_prev_rel_time"],
                noisy_cov_init_time=config["ebop_algorithm"]["noisy_cov_init_time"],
                registry_HA_threshold=config["ebop_algorithm"]["registry_HA_threshold"],
                rtf_estimator=rtf_estimator,
                max_sources=config["dataset"]["max_sources"],
                interferer_gain=config["ebop_algorithm"]["interferer_gain"],
                source_activity_method=None, # Use Oracle from batch
                batch_size=1,
                loss_config=config["framework"]["rtf_loss"],
                optimizer_config=config["framework"]["rtf_optimizer"],
                lr_scheduler_config=config["framework"]["rtf_lr_scheduler"],
                compute_complexity_metrics=False,
                check_causality=False,
            )
        else:
            print(f"Skipping {rtf_method_name}: Class not found.")
            continue

    # --- Run Predict ---    
    trainer = pl.Trainer(
        accelerator=config["framework"]["rtf_accelerator"],
        devices=config["framework"]["rtf_devices"],
        precision=config["framework"]["rtf_precision"],
        logger=False,
        enable_checkpointing=False,
        callbacks=[rtf_callback],
        inference_mode=True
    )

    print(f"Running predictions for {rtf_method_name}...")
    # Add return_predictions=False to avoid accumulating results in memory
    trainer.predict(rtf_model, dataloaders=test_loader, return_predictions=False)
    
    # --- Print Stats ---
    rtf_callback.print_stats(rtf_method_name)
    results[rtf_method_name] = rtf_callback.rtfs.copy()

    # Cleanup
    del rtf_model
    del trainer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3]


Benchmarking methods: ['CWv', 'BOP', 'BOP-S', 'BOP-W', 'BOPO', 'BOPO-S', 'BOPO-W', 'block-online-GSS']

Preparing CWv...
Running predictions for CWv...


Predicting: |          | 0/? [00:00<?, ?it/s]


--- RTF Statistics for CWv ---
Min:    0.005143
Max:    0.062316
Mean:   0.007203
Median: 0.007139
Std:    0.001260
Count:  2640
---------------------------------------



GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3]


...


--- RTF Statistics for BOPO-W ---
Min:    0.005778
Max:    0.058921
Mean:   0.008221
Median: 0.008136
Std:    0.001422
Count:  2640
---------------------------------------


Preparing block-online-GSS...


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3]


Running predictions for block-online-GSS...


Predicting: |          | 0/? [00:00<?, ?it/s]


--- RTF Statistics for block-online-GSS ---
Min:    0.081959
Max:    0.277887
Mean:   0.143052
Median: 0.142791
Std:    0.020113
Count:  2640
---------------------------------------

